In [1]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA')\
                  .named('template_pool')

mutated_pool = template_pool.stylize('cre', style='goldenrod')\
                            .mutagenize('cre',
                                        mutation_rate=0.1, 
                                        style_mutations='yellow bold',
                                        mode='random', 
                                        num_states=5, 
                                        prefix='mut').named('mutated_pool')

deletion_pool = template_pool.stylize('cre', style='salmon')\
                             .deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style_deletion='red bold',
                                            prefix='del').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        iter_order=-1,
                        prefix='site').named('sites_pool')

insertion_pool = template_pool.stylize('cre', style='blue')\
                              .insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              prefix='ins',
                                              prefix_position='pos', 
                                              prefix_insert='site',
                                              style_insertion='cyan bold').named('insertion_pool')
                              
shuffle_pool = template_pool.stylize('cre', style='purple')\
                            .shuffle_scan('cre', 
                                          shuffle_length=5, 
                                          positions=slice(None, None, 5), 
                                          mode='sequential', 
                                          style_shuffle='magenta')

combo_pool = pp.stack([mutated_pool, deletion_pool, shuffle_pool]).named('stack_pool')\
    .repeat_states(2, prefix='v', iter_order=-2).named('repeated_pool')\
    .insert_kmers('bc', mode='random', length=5, prefix='bc', style_kmers='green')\
    .named('combo_pool').stylize(which='tags', style='gray')

combo_pool.print_library(show_name=True,seed=12)

pool[15]: seq_length=None, num_states=30
state  name             seq
    0  mut_0.v_0.bc_0   TCCCGACT<cre>GAGAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>AGGCC</bc>AGATCGGA
    1  mut_0.v_1.bc_1   TCCCGACT<cre>GAGAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>TGCGG</bc>AGATCGGA
    2  mut_1.v_0.bc_2   TCCCGACT<cre>GGAAAGCGGGTAGTGAGCACACTGG</cre>ATTACGG<bc>TTCGG</bc>AGATCGGA
    3  mut_1.v_1.bc_3   TCCCGACT<cre>GGAAAGCGGGTAGTGAGCACACTGG</cre>ATTACGG<bc>TTGCA</bc>AGATCGGA
    4  mut_2.v_0.bc_4   TCCCGACT<cre>GGAAAGCGGTCAGTAAGCACATAGG</cre>ATTACGG<bc>CGCGG</bc>AGATCGGA
    5  mut_2.v_1.bc_5   TCCCGACT<cre>GGAAAGCGGTCAGTAAGCACATAGG</cre>ATTACGG<bc>GCGCA</bc>AGATCGGA
    6  mut_3.v_0.bc_6   TCCCGACT<cre>GGAAAGCGGGCAGGGAGCATACTGG</cre>ATTACGG<bc>ATATG</bc>AGATCGGA
    7  mut_3.v_1.bc_7   TCCCGACT<cre>GGAAAGCGGGCAGGGAGCATACTGG</cre>ATTACGG<bc>TGGAG</bc>AGATCGGA
    8  mut_4.v_0.bc_8   TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>TATAC</bc>AGATCGGA
    9  mut_4.v_1.bc_9   TCCCGACT<cre>GGAAAGCGGGCA

Pool(id=15, name='pool[15]', op='op[15]:stylize', num_states=30)

In [2]:
combo_pool.print_dag()

pool[15] (pool, n=30)
└── op[15]:stylize [mode=fixed, n=1]
    └── combo_pool (pool, n=30)
        └── op[14]:get_kmers [mode=random, n=30]
            └── repeated_pool (pool, n=30)
                └── op[13]:repeat [mode=sequential, n=2]
                    └── stack_pool (pool, n=15)
                        └── op[12]:stack [mode=sequential, n=3]
                            ├── mutated_pool (pool, n=5)
                            │   └── op[2]:mutagenize [mode=random, n=5]
                            │       └── pool[1] (pool, n=1)
                            │           └── op[1]:stylize [mode=fixed, n=1]
                            │               └── template_pool (pool, n=1)
                            │                   └── op[0]:from_seq [mode=fixed, n=1]
                            ├── deletion_pool (pool, n=5)
                            │   └── op[4]:deletion_scan [mode=sequential, n=5]
                            │       └── pool[3] (pool, n=1)
                           

Pool(id=15, name='pool[15]', op='op[15]:stylize', num_states=30)

In [3]:
combo_pool.generate_library?